# Vie-GameEmo — Stage 0: Chuẩn bị Dataset

**Notebook này thực hiện:**
1. Tải video từ YouTube (hoặc dùng video có sẵn)
2. **Cắt video thô thành các clip ngắn 3-7 giây** (mỗi clip là một đơn vị gán nhãn)
3. Tiền xử lý: tách audio, trích xuất frames, phát hiện webcam
4. Gán nhãn đa tác tử: Whisper ASR → OpenFace AUs → Qwen-VL → Qwen-Audio → Consolidator
5. **Mỗi nhãn đi kèm `confidence` (0.0–1.0) — cả nhãn thủ công lẫn nhãn do Consolidator dự đoán**
6. Xuất annotations JSON ra Kaggle output

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 x1** (hoặc P100)
- Internet: **BẬT** (Settings → Internet)
- Runtime: ~4-6 giờ cho 50 clips

---
⚠️ **Stage 0 chạy độc lập.** Sau khi xong, download `data/annotations/` và dùng cho notebook Training.

In [3]:
# ============================================================
# CELL 1 — Kiểm tra môi trường
# ============================================================
import os, sys, subprocess

IS_KAGGLE = os.path.exists('/kaggle')
WORKING = '/kaggle/working' if IS_KAGGLE else '/tmp/vie-gameemo'
os.makedirs(WORKING, exist_ok=True)
print(f'Platform : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Working  : {WORKING}')

# GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU      : {gpu.name} | {gpu.total_memory / 1e9:.1f} GB VRAM')
else:
    print('⚠️  Không có GPU — hãy bật Accelerator trong Settings')

Platform : Kaggle
Working  : /kaggle/working
GPU      : Tesla T4 | 15.6 GB VRAM


In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH (chỉnh tại đây)
# ============================================================

# --- Nguồn dữ liệu ---
# 'youtube'  : tải từ danh sách URL bên dưới
# 'existing' : dùng video có sẵn trong /kaggle/input/<dataset_name>/
DATA_SOURCE = 'youtube'   # <-- ĐỔI TẠI ĐÂY

# Nếu DATA_SOURCE = 'existing', tên Kaggle dataset chứa video
EXISTING_DATASET_PATH = '/kaggle/input/vie-gameemo-videos'  # mount từ Kaggle Dataset

# --- Danh sách URL YouTube (dùng khi DATA_SOURCE = 'youtube') ---
YOUTUBE_URLS = [
    # Thêm URL YouTube livestream/clip của streamer vào đây
    # 'https://www.youtube.com/watch?v=XXXXXXXXX',
    'https://www.youtube.com/live/LhWXapOorJs?si=sDtL3xH6RP7J69M6'
]

# --- Cắt video thô thành clip ngắn 3–7 giây ---
# Mỗi video thô (đã tải hoặc input) sẽ được cắt thành nhiều đoạn liên tiếp dài
# CLIP_TARGET_DURATION giây. Đoạn cuối nếu < CLIP_MIN_DURATION sẽ bị bỏ;
# nếu nằm trong [CLIP_MIN_DURATION, CLIP_MAX_DURATION] thì giữ nguyên.
CLIP_TARGET_DURATION = 5.0   # giây (mặc định 5s — giữa khoảng 3–7)
CLIP_MIN_DURATION    = 4.5
CLIP_MAX_DURATION    = 5.5
# Nếu muốn bỏ qua N giây đầu/cuối mỗi video thô (intro/outro), chỉnh ở đây:
RAW_TRIM_HEAD_SEC = 600
RAW_TRIM_TAIL_SEC = 0.0
# Giới hạn tổng số segment tạo ra trên mỗi video thô (None = không giới hạn)
MAX_SEGMENTS_PER_VIDEO = 200

assert CLIP_MIN_DURATION <= CLIP_TARGET_DURATION <= CLIP_MAX_DURATION, \
    'Phải có CLIP_MIN_DURATION <= CLIP_TARGET_DURATION <= CLIP_MAX_DURATION'

# --- Model annotation ---
# Chọn model cho Consolidator (annotation reasoning)
# 'Qwen/Qwen2.5-7B-Instruct'   → T4 OK, chất lượng tốt (khuyến nghị)
# 'Qwen/Qwen2.5-1.5B-Instruct' → T4 thoải mái, nhanh hơn ~3x, chất lượng thấp hơn
# 'Qwen/Qwen2.5-32B-Instruct'  → cần A100, chất lượng cao nhất
# 'Qwen/Qwen3-8B'              → Qwen3 mới nhất, hỗ trợ thinking mode, T4 OK (4bit)
ANNOTATION_MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # <-- ĐỔI TẠI ĐÂY

# Model Qwen-VL cho visual descriptions
QWEN_VL_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'   # hoặc 'Qwen/Qwen2.5-VL-3B-Instruct'

# Model Qwen-Audio cho audio descriptions
QWEN_AUDIO_MODEL = 'Qwen/Qwen2-Audio-7B-Instruct'

# Quantization (4bit tiết kiệm VRAM nhất)
QUANTIZATION = '4bit'   # '4bit' | '8bit' | 'none'
# --- ASR Backend ---
# 'whisper'    : faster-whisper (openai/whisper-large-v3) — tốt cho đa ngôn ngữ
# 'phowhisper' : VinAI PhoWhisper — fine-tuned tiếng Việt, tốt hơn cho giọng VN thuần
ASR_BACKEND = 'whisper'   # <-- ĐỔI TẠI ĐÂY

# Model Whisper (khi ASR_BACKEND = 'whisper')
WHISPER_MODEL = 'openai/whisper-large-v3'
#   Nhanh hơn: 'openai/whisper-large-v3-turbo'  (~6x nhanh, quality gần tương đương)
#   Nhẹ hơn:  'openai/whisper-medium'

# Model PhoWhisper (khi ASR_BACKEND = 'phowhisper')
PHOWHISPER_MODEL = 'vinai/PhoWhisper-large'
#   Nhẹ hơn: 'vinai/PhoWhisper-medium' | 'vinai/PhoWhisper-small'

# BARTpho post-processing (thêm ~3 GB VRAM + ~5-10s/clip)
# Bật nếu ASR output nhiều lỗi từ ghép / thiếu dấu câu
USE_BARTPHO = False
BARTPHO_MODEL = 'vinai/bartpho-syllable-1_5'
#   Nặng hơn + tốt hơn: 'vinai/bartpho-word'



# --- Gán nhãn cảm xúc thủ công (schema gaming_8) + tỉ lệ chắc chắn ---
# Định dạng linh hoạt cho mỗi clip:
#   1) Chuỗi nhãn:                     'hype'                        → confidence = 1.0
#   2) Tuple (nhãn, confidence):       ('hype', 0.85)
#   3) Dict đầy đủ:                    {'label': 'hype', 'confidence': 0.85,
#                                       'alternatives': {'amused': 0.10, 'shocked': 0.05}}
# confidence ∈ [0.0, 1.0] — mức tin cậy chủ quan của người gán nhãn.
# 'alternatives' (tuỳ chọn) là các nhãn thay thế kèm xác suất; khi có, tổng
# alternatives + confidence nên ~ 1.0 (sẽ được tự chuẩn hoá).
#
# Lưu ý quan trọng: clip_id phải khớp với clip ĐÃ CẮT 3–7s (xem CELL 7 — sẽ in
# ra danh sách clip_id sau khi cắt). Chạy notebook một lần để sinh clip rồi mới
# điền nhãn vào đây và chạy lại.
#
# Nhãn hợp lệ (xem docs/annotation_guideline.md):
#   neutral  | hype     | amused  | tilted
#   sad      | shocked  | fear    | disgusted
CLIP_LABELS = {
   
}

# Confidence ngầm định cho clip không có entry trong CLIP_LABELS (placeholder 'neutral').
# Đặt thấp để dễ lọc khi training.
DEFAULT_PLACEHOLDER_CONFIDENCE = 0.0

# --- Giới hạn số clip SAU KHI CẮT (để test nhanh) ---
MAX_CLIPS = 20    # None = không giới hạn

# --- Đường dẫn ---
DATA_DIR     = os.path.join(WORKING, 'data')
RAW_DIR      = os.path.join(DATA_DIR, 'raw_videos')          # video gốc trước khi cắt
SEG_DIR      = os.path.join(DATA_DIR, 'clips')               # clip đã cắt 3–7s
PROC_DIR     = os.path.join(DATA_DIR, 'processed')
ANNOT_DIR    = os.path.join(DATA_DIR, 'annotations')
PROJECT_DIR  = os.path.join(WORKING, 'vie-gameemo-skeleton')

for d in [DATA_DIR, RAW_DIR, SEG_DIR, PROC_DIR, ANNOT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config OK')
print(f'  DATA_SOURCE          : {DATA_SOURCE}')
print(f'  ANNOTATION_MODEL     : {ANNOTATION_MODEL}')
print(f'  CLIP_TARGET_DURATION : {CLIP_TARGET_DURATION}s  (bounds [{CLIP_MIN_DURATION}, {CLIP_MAX_DURATION}])')
print(f'  MAX_CLIPS            : {MAX_CLIPS}')

Config OK
  DATA_SOURCE          : youtube
  ANNOTATION_MODEL     : Qwen/Qwen2.5-7B-Instruct
  CLIP_TARGET_DURATION : 5.0s  (bounds [3.0, 7.0])
  MAX_CLIPS            : 20


In [5]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
# Kaggle có sẵn: torch, numpy, opencv, PIL, ffmpeg
# Cần cài thêm:
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    peft>=0.13.0 \
    bitsandbytes>=0.43.0 \
    faster-whisper>=1.0.3 \
    librosa>=0.10.1 \
    soundfile \
    mediapipe\
    yt-dlp>=2024.10.0 \
    pydantic>=2.8.0 \
    pyyaml>=6.0.2 \
    scikit-learn>=1.5.0 \
    qwen-vl-utils

# Thêm Qwen3 support nếu cần
if 'Qwen3' in ANNOTATION_MODEL:
    !pip install -q transformers --upgrade

print('Cài đặt hoàn tất')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
Cài đặt hoàn tất


In [6]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import subprocess, shutil

# Option A: project có sẵn trong Kaggle input dataset (thêm 'vie-gameemo-code' vào notebook inputs)
CODE_INPUT = '/kaggle/input/vie-gameemo-code'

if os.path.exists(CODE_INPUT):
    # Copy sang working (cần write permission)
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(CODE_INPUT, PROJECT_DIR)
    print(f'Project loaded from Kaggle input: {CODE_INPUT}')

elif os.path.exists('/kaggle/working/vie-gameemo-skeleton'):
    PROJECT_DIR = '/kaggle/working/vie-gameemo-skeleton'
    print(f'Project đã có tại: {PROJECT_DIR}')

else:
    # Option B: clone từ GitHub (cần internet)
    # Thay GITHUB_URL bằng repo thực tế của bạn
    GITHUB_URL = 'https://github.com/rhy221/vie_gameemo.git'
    result = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('⚠️  Git clone thất bại. Hãy thêm project code vào Kaggle input dataset.')
        print(result.stderr)
    else:
        print(f'Cloned project → {PROJECT_DIR}')

# Thêm src vào Python path
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
    print(f'Added to sys.path: {SRC_DIR}')

# Tạo config.yaml tối giản cho Stage 0
CONFIG_PATH = os.path.join(WORKING, 'config_stage0.yaml')
CONFIG_CONTENT = f"""
seed: 42
logging:
  level: INFO
  file: {WORKING}/logs/stage0.log
  console: true
paths:
  data_root: {DATA_DIR}
  raw_videos: {RAW_DIR}
  processed: {PROC_DIR}
  audios: {PROC_DIR}/audios
  frames: {PROC_DIR}/frames
  faces: {PROC_DIR}/faces
  contexts: {PROC_DIR}/contexts
  annotations: {ANNOT_DIR}
  features: {DATA_DIR}/features
  checkpoints: {WORKING}/checkpoints
  results: {WORKING}/results
annotation:
  openface:
    binary_path: {WORKING}/OpenFace/build/bin/FeatureExtraction
  whisper_asr:
    model_name: openai/whisper-large-v3
    backend: faster-whisper
    compute_type: float16
    language: vi
    vad_filter: true
  qwen_vl:
    model_name: {QWEN_VL_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 200
  qwen_audio:
    model_name: {QWEN_AUDIO_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 200
  consolidator:
    model_name: {ANNOTATION_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 400
  batch_size: 8
preprocess:
  audio:
    sample_rate: 16000
  frames:
    fps: 2
    format: jpg
compute:
  profile: colab
  annotation_vram_gb: 15
"""

os.makedirs(os.path.join(WORKING, 'logs'), exist_ok=True)
with open(CONFIG_PATH, 'w') as f:
    f.write(CONFIG_CONTENT)
print(f'Config → {CONFIG_PATH}')

Cloned project → /kaggle/working/vie-gameemo-skeleton
Added to sys.path: /kaggle/working/vie-gameemo-skeleton/src
Config → /kaggle/working/config_stage0.yaml


In [7]:
# ============================================================
# CELL 5 — Build OpenFace (cần ~15 phút, chạy một lần)
# ============================================================
import os
import subprocess
import shutil

OPENFACE_DIR = os.path.join(WORKING, 'OpenFace')
DLIB_DIR = os.path.join(WORKING, 'dlib')
OPENFACE_BINARY = os.path.join(OPENFACE_DIR, 'build/bin/FeatureExtraction')
USE_OPENFACE = False

# 1. Dọn dẹp sạch sẽ các thư mục cũ để tránh lỗi git clone
for folder in [OPENFACE_DIR, DLIB_DIR]:
    if os.path.exists(folder):
        print(f"🧹 Đang xóa thư mục cũ: {folder}...")
        subprocess.run(f"sudo rm -rf {folder}", shell=True)

print('🚀 Bắt đầu quá trình build (có thể mất ~15 phút)...')

BUILD_CMDS = [
    'sudo apt-get update -y -q',
    'sudo apt-get install -y -q libopencv-dev cmake libboost-all-dev libx11-dev',

    # Build Dlib từ source
    f'git clone --depth=1 https://github.com/davisking/dlib.git {DLIB_DIR}',
    f'mkdir -p {DLIB_DIR}/build',
    f'cmake -S {DLIB_DIR} -B {DLIB_DIR}/build -DCMAKE_BUILD_TYPE=Release',
    f'cmake --build {DLIB_DIR}/build --parallel 2',
    f'sudo cmake --install {DLIB_DIR}/build',

    # Build OpenFace
    f'git clone --depth=1 https://github.com/TadasBaltrusaitis/OpenFace.git {OPENFACE_DIR}',
    f'bash {OPENFACE_DIR}/download_models.sh',
    f'mkdir -p {OPENFACE_DIR}/build',
    f'cmake -B {OPENFACE_DIR}/build -S {OPENFACE_DIR} -DCMAKE_BUILD_TYPE=Release -Wno-dev',
    f'cmake --build {OPENFACE_DIR}/build --parallel 2',
]

success = True
for cmd in BUILD_CMDS:
    print(f"⚙️ Đang chạy: {cmd[:50]}...")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'⚠️ Lệnh thất bại: {cmd}')
        print(r.stderr[-500:] if r.stderr else 'Không có thông báo lỗi chi tiết.')
        success = False
        break

if success and os.path.exists(OPENFACE_BINARY):
    USE_OPENFACE = True
    print('✅ OpenFace build thành công!')
else:
    print('⚠️ OpenFace build thất bại → Hệ thống sẽ dùng MediaPipe làm fallback.')

print(f'USE_OPENFACE = {USE_OPENFACE}')

🚀 Bắt đầu quá trình build (có thể mất ~15 phút)...
⚙️ Đang chạy: sudo apt-get update -y -q...
⚙️ Đang chạy: sudo apt-get install -y -q libopencv-dev cmake lib...
⚙️ Đang chạy: git clone --depth=1 https://github.com/davisking/d...
⚙️ Đang chạy: mkdir -p /kaggle/working/dlib/build...
⚙️ Đang chạy: cmake -S /kaggle/working/dlib -B /kaggle/working/d...
⚙️ Đang chạy: cmake --build /kaggle/working/dlib/build --paralle...
⚙️ Đang chạy: sudo cmake --install /kaggle/working/dlib/build...
⚙️ Đang chạy: git clone --depth=1 https://github.com/TadasBaltru...
⚙️ Đang chạy: bash /kaggle/working/OpenFace/download_models.sh...
⚙️ Đang chạy: mkdir -p /kaggle/working/OpenFace/build...
⚙️ Đang chạy: cmake -B /kaggle/working/OpenFace/build -S /kaggle...
⚙️ Đang chạy: cmake --build /kaggle/working/OpenFace/build --par...
✅ OpenFace build thành công!
USE_OPENFACE = True


## Bước 1 — Tải video THÔ và cắt thành clip 3–7 giây

- **CELL 6**: tải/copy video gốc vào `data/raw_videos/`
- **CELL 7**: dùng ffmpeg cắt mỗi video gốc thành nhiều đoạn `CLIP_TARGET_DURATION` giây (mỗi đoạn ∈ `[CLIP_MIN_DURATION, CLIP_MAX_DURATION]` giây). Mỗi đoạn = 1 clip để gán nhãn.
- **CELL 8**: tạo `labels.csv` cho danh sách clip — kèm cột `confidence` (mức chắc chắn của nhãn thủ công).

Sau khi chạy lần đầu, copy danh sách `clip_id` từ output CELL 7 vào `CLIP_LABELS` (CELL 2) kèm confidence rồi chạy lại.

In [8]:
# ============================================================
# CELL 6 — Tải video THÔ (chưa cắt) vào RAW_DIR
# Mỗi mục trong DATA_SOURCE -> một file video gốc; sẽ được cắt ở CELL 7.
# ============================================================
raw_video_paths = []   # video gốc, chưa cắt

if DATA_SOURCE == 'youtube':
    if not YOUTUBE_URLS:
        print('⚠️  YOUTUBE_URLS rỗng — thêm URL vào CELL 2 và chạy lại')
    else:
        for i, url in enumerate(YOUTUBE_URLS):
            raw_id = f'raw_{i+1:03d}'
            out_path = os.path.join(RAW_DIR, f'{raw_id}.mp4')
            if os.path.exists(out_path):
                print(f'  Skip (đã có): {out_path}')
                raw_video_paths.append(out_path)
                continue
            print(f'  Downloading {raw_id}: {url[:60]}...')
            r = subprocess.run([
                'yt-dlp', url,
                '-f', 'best[height<=720]',
                '--merge-output-format', 'mp4',
                '-o', out_path,
                '--no-playlist',
                '--quiet', '--progress',
            ], capture_output=False)
            if r.returncode == 0 and os.path.exists(out_path):
                raw_video_paths.append(out_path)
                print(f'  ✅ {out_path}')
            else:
                print(f'  ❌ Tải thất bại: {url}')

elif DATA_SOURCE == 'existing':
    if not os.path.exists(EXISTING_DATASET_PATH):
        print(f'⚠️  Không tìm thấy: {EXISTING_DATASET_PATH}')
        print('Hãy thêm dataset video vào notebook inputs (Add data → Your datasets)')
    else:
        import glob
        found = sorted(glob.glob(os.path.join(EXISTING_DATASET_PATH, '**', '*.mp4'), recursive=True))
        for src in found:
            dst = os.path.join(RAW_DIR, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            raw_video_paths.append(dst)
        print(f'Loaded {len(raw_video_paths)} video(s) từ {EXISTING_DATASET_PATH}')

print(f'\nTổng video THÔ (chưa cắt): {len(raw_video_paths)}')
for p in raw_video_paths[:5]:
    print(f'  {p}')

  ✅ /kaggle/working/data/raw_videos/raw_001.mp4            

Tổng video THÔ (chưa cắt): 1
  /kaggle/working/data/raw_videos/raw_001.mp4


In [9]:
# ============================================================
# CELL 7 — Cắt mỗi video THÔ thành các clip ngắn 3–7 giây
# Dùng ffmpeg với -c copy (stream copy) để cắt cực nhanh, không re-encode.
# Mỗi segment có tên: {raw_stem}_seg_{idx:03d}.mp4  (đây chính là clip_id)
# ============================================================
import json
import math
from pathlib import Path

def _probe_duration(video_path: str) -> float:
    """Trả về độ dài video (giây) qua ffprobe."""
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', video_path],
        capture_output=True, text=True,
    )
    try:
        return float(r.stdout.strip())
    except (TypeError, ValueError):
        return 0.0

def _cut_segment(src: str, dst: str, start: float, duration: float) -> bool:
    """Cắt một đoạn từ src lưu vào dst. Trả về True nếu thành công."""
    if os.path.exists(dst):
        return True
    cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-ss', f'{start:.3f}', '-i', src,
        '-t', f'{duration:.3f}',
        # re-encode để keyframe-accurate và không vỡ khi -ss < keyframe.
        # Tốc độ vẫn nhanh với preset ultrafast trên clip ngắn.
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-c:a', 'aac', '-b:a', '128k',
        '-movflags', '+faststart',
        dst,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0 and os.path.exists(dst)

video_paths = []   # danh sách clip 3–7s SAU KHI CẮT — các cell sau dùng biến này
segment_meta = {}  # clip_id -> {raw_video, start_sec, duration_sec}

if not raw_video_paths:
    print('⚠️  Chưa có video thô — chạy CELL 6 trước.')
else:
    for raw_path in raw_video_paths:
        raw_stem = Path(raw_path).stem
        total = _probe_duration(raw_path)
        if total <= 0:
            print(f'  ⚠️  Không đọc được duration: {raw_path}')
            continue

        start = max(0.0, RAW_TRIM_HEAD_SEC)
        end   = max(start, total - max(0.0, RAW_TRIM_TAIL_SEC))
        usable = end - start
        if usable < CLIP_MIN_DURATION:
            print(f'  ⚠️  {raw_stem}: video quá ngắn ({usable:.1f}s < {CLIP_MIN_DURATION}s)')
            continue

        # Số segment đầy đủ với độ dài target
        n_full = int(usable // CLIP_TARGET_DURATION)
        tail   = usable - n_full * CLIP_TARGET_DURATION

        seg_specs = [(start + k * CLIP_TARGET_DURATION, CLIP_TARGET_DURATION)
                     for k in range(n_full)]
        # Đoạn cuối: chỉ giữ nếu nằm trong [MIN, MAX]
        if CLIP_MIN_DURATION <= tail <= CLIP_MAX_DURATION:
            seg_specs.append((start + n_full * CLIP_TARGET_DURATION, tail))

        if MAX_SEGMENTS_PER_VIDEO:
            seg_specs = seg_specs[:MAX_SEGMENTS_PER_VIDEO]

        kept = 0
        for idx, (s, d) in enumerate(seg_specs, start=1):
            # Ràng buộc cứng [MIN, MAX]
            if not (CLIP_MIN_DURATION <= d <= CLIP_MAX_DURATION):
                continue
            clip_id = f'{raw_stem}_seg_{idx:03d}'
            dst = os.path.join(SEG_DIR, f'{clip_id}.mp4')
            ok = _cut_segment(raw_path, dst, s, d)
            if ok:
                video_paths.append(dst)
                segment_meta[clip_id] = {
                    'raw_video': raw_path,
                    'start_sec': round(s, 3),
                    'duration_sec': round(d, 3),
                }
                kept += 1
            else:
                print(f'    ❌ Cắt thất bại: {clip_id} ({s:.1f}s, {d:.1f}s)')
        print(f'  {raw_stem}: {kept}/{len(seg_specs)} segment (total {total:.1f}s, usable {usable:.1f}s)')

    # Giới hạn MAX_CLIPS sau khi cắt
    if MAX_CLIPS and len(video_paths) > MAX_CLIPS:
        kept_ids = {os.path.splitext(os.path.basename(p))[0] for p in video_paths[:MAX_CLIPS]}
        video_paths = video_paths[:MAX_CLIPS]
        segment_meta = {k: v for k, v in segment_meta.items() if k in kept_ids}

# Lưu metadata để các stage sau (training, eval) biết clip lấy từ đâu
meta_path = os.path.join(SEG_DIR, 'segments.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(segment_meta, f, ensure_ascii=False, indent=2)

print(f'\n✅ Tổng clip 3–{int(CLIP_MAX_DURATION)}s đã cắt: {len(video_paths)}')
print(f'   Metadata → {meta_path}')
print('\nCác clip_id (copy vào CLIP_LABELS ở CELL 2 để gán nhãn):')
for p in video_paths:
    cid = os.path.splitext(os.path.basename(p))[0]
    d = segment_meta[cid]['duration_sec']
    print(f"    '{cid}': ('neutral', 0.5),   # {d:.2f}s")

  raw_001: 200/200 segment (total 11776.3s, usable 11176.3s)

✅ Tổng clip 3–7s đã cắt: 20
   Metadata → /kaggle/working/data/clips/segments.json

Các clip_id (copy vào CLIP_LABELS ở CELL 2 để gán nhãn):
    'raw_001_seg_001': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_002': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_003': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_004': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_005': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_006': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_007': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_008': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_009': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_010': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_011': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_012': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_013': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_014': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_015': ('neutral', 0.5),   # 5.00s
    'raw_001_seg_016': ('neutral

In [10]:
# ============================================================
# CELL 8 — Tạo labels CSV (schema gaming_8) + confidence
# Điền nhãn thủ công vào CLIP_LABELS ở CELL 2, hoặc để placeholder.
# Hỗ trợ 3 định dạng entry: str | (label, conf) | dict
# ============================================================
import csv
import json

# Schema gaming_8 — order khớp EmotionLabel enum trong src/vie_gameemo/data/schemas.py
VALID_LABELS = [
    'neutral', 'hype', 'amused', 'tilted', 'sad', 'shocked', 'fear', 'disgusted',
]
LABELS_CSV = os.path.join(ANNOT_DIR, 'labels.csv')


def _normalize_label_entry(clip_id: str, entry):
    """Chuẩn hoá entry trong CLIP_LABELS thành (label, confidence, alternatives_json).

    - alternatives_json là string JSON của dict {label: prob} (có thể rỗng "{}").
    - confidence sẽ được clamp về [0, 1].
    - Nếu có alternatives, chuẩn hoá để tổng (confidence + alternatives) = 1.
    """
    alternatives: dict[str, float] = {}
    explicit = True   # có entry trong CLIP_LABELS hay không

    if entry is None:
        label = 'neutral'
        confidence = float(DEFAULT_PLACEHOLDER_CONFIDENCE)
        explicit = False
    elif isinstance(entry, str):
        label = entry
        confidence = 1.0
    elif isinstance(entry, tuple) and len(entry) == 2:
        label, confidence = entry[0], float(entry[1])
    elif isinstance(entry, dict):
        label = entry.get('label', 'neutral')
        confidence = float(entry.get('confidence', 1.0))
        alternatives = {k: float(v) for k, v in entry.get('alternatives', {}).items()}
    else:
        raise ValueError(
            f'Định dạng nhãn không hợp lệ cho {clip_id!r}: {entry!r}. '
            'Dùng str | (label, confidence) | dict.'
        )

    if label not in VALID_LABELS:
        raise ValueError(
            f'Nhãn không hợp lệ cho {clip_id!r}: {label!r}. Phải thuộc {VALID_LABELS}'
        )
    for alt in alternatives:
        if alt not in VALID_LABELS:
            raise ValueError(
                f'Nhãn thay thế không hợp lệ cho {clip_id!r}: {alt!r}. Phải thuộc {VALID_LABELS}'
            )

    confidence = max(0.0, min(1.0, confidence))

    # Chuẩn hoá tổng (confidence + alternatives) = 1 khi có alternatives
    if alternatives:
        total = confidence + sum(alternatives.values())
        if total > 0:
            confidence = confidence / total
            alternatives = {k: v / total for k, v in alternatives.items()}

    return label, confidence, alternatives, explicit


rows = []
manual_label_meta = {}   # dùng cho cell sau (consolidator + JSON)
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    label, conf, alts, explicit = _normalize_label_entry(clip_id, CLIP_LABELS.get(clip_id))
    rows.append({
        'clip_id': clip_id,
        'emotion_label': label,
        'confidence': f'{conf:.4f}',
        'alternatives': json.dumps(alts, ensure_ascii=False),
        'is_placeholder': '0' if explicit else '1',
        'video_path': vp,
    })
    manual_label_meta[clip_id] = {
        'label': label,
        'confidence': conf,
        'alternatives': alts,
        'is_placeholder': not explicit,
    }

with open(LABELS_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['clip_id', 'emotion_label', 'confidence',
                    'alternatives', 'is_placeholder', 'video_path'],
    )
    writer.writeheader()
    writer.writerows(rows)

n_placeholder = sum(1 for m in manual_label_meta.values() if m['is_placeholder'])
print(f'Labels CSV: {LABELS_CSV}')
print(f'Số clip   : {len(rows)}  ({n_placeholder} placeholder, {len(rows) - n_placeholder} có nhãn thủ công)')
if rows:
    avg_conf = sum(float(r['confidence']) for r in rows) / len(rows)
    print(f'Confidence trung bình (manual): {avg_conf:.3f}')
if n_placeholder:
    print('⚠️  Có clip placeholder — điền nhãn + confidence vào CLIP_LABELS ở CELL 2 rồi chạy lại để cải thiện chất lượng.')

Labels CSV: /kaggle/working/data/annotations/labels.csv
Số clip   : 20  (0 placeholder, 20 có nhãn thủ công)
Confidence trung bình (manual): 0.500


## Bước 2 — Tiền xử lý video

In [11]:
# ============================================================
# CELL 8 — Tách audio + frames
# ============================================================
# Kiểm tra ffmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True)
print('ffmpeg:', 'OK' if r.returncode == 0 else '❌ KHÔNG TÌM THẤY')

sys.path.insert(0, SRC_DIR) if SRC_DIR not in sys.path else None

from vie_gameemo.preprocess.demux import extract_audio, extract_frames
from pathlib import Path
import inspect

print(inspect.signature(extract_frames))

for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_dir = os.path.join(PROC_DIR, 'audios')
    frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(frames_dir, exist_ok=True)

    audio_path = os.path.join(audio_dir, f'{clip_id}.wav')
    if not os.path.exists(audio_path):
        extract_audio(Path(vp), Path(audio_path))
        print(f'  Audio → {audio_path}')
    else:
        print(f'  Skip audio (đã có): {clip_id}')

    if not os.listdir(frames_dir):
        n = extract_frames(Path(vp), Path(frames_dir), fps=4)
        print(f'  Frames → {frames_dir} ({n} frames)')
    else:
        print(f'  Skip frames (đã có): {clip_id}')

print('\n✅ Tiền xử lý xong')

ffmpeg: OK
(video_path: pathlib.Path, output_dir: pathlib.Path, fps: float = 2.0, quality: int = 90) -> list[pathlib.Path]
  Audio → /kaggle/working/data/processed/audios/raw_001_seg_001.wav
  Frames → /kaggle/working/data/processed/frames/raw_001_seg_001 ([PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0000.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0001.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0002.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0003.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0004.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0005.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0006.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0007.jpg'), PosixPath('/kaggle/working/data/processed/frames/raw_001_seg_001/frame_0008.jp

In [17]:
# ============================================================
# CELL 9 — Phát hiện vùng webcam (YOLOv8-pose + DBSCAN)
# ============================================================
# Dùng yolov8n-pose.pt (model CHÍNH THỨC của Ultralytics, auto-download từ
# Ultralytics CDN — không bị 404 như akanametov/yolov8-face GitHub release
# vốn hay bị chặn trên Kaggle). Pose model cho 17 keypoints; ta lấy 5 điểm
# trên mặt (mũi, 2 mắt, 2 tai) để derive face bbox, rồi DBSCAN cluster như cũ.
import os
import sys
import json
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path

import cv2
import numpy as np


def _ensure_ultralytics():
    try:
        from ultralytics import YOLO  # noqa: F401
        return True
    except ImportError:
        pass
    print("Đang cài ultralytics...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("Cài ultralytics thất bại. stderr:")
        print(r.stderr[-1500:])
        return False
    try:
        from ultralytics import YOLO  # noqa: F401
        return True
    except Exception as e:
        print(f"Vẫn không import được ultralytics: {e}")
        return False


@dataclass
class _BBox:
    xmin: float
    ymin: float
    width: float
    height: float
    stability_score: float
    edge_distance: float


def _sample_frames(clip_path: Path, n_frames: int = 30):
    cap = cv2.VideoCapture(str(clip_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    n = min(n_frames, total)
    indices = np.linspace(0, total - 1, n, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, f = cap.read()
        if ret:
            frames.append(f)
    cap.release()
    return frames


def _pose_detect_faces(model, frames, conf: float = 0.3, pad: float = 0.6):
    """Suy ra face bbox normalized [0,1] từ 5 keypoints đầu (nose, eyes, ears)
    của yolov8-pose. `pad` = tỉ lệ mở rộng bbox quanh các keypoints."""
    if not frames:
        return []
    detections = []
    results = model.predict(frames, conf=conf, verbose=False)
    for frame, res in zip(frames, results):
        if res.keypoints is None or len(res.keypoints) == 0:
            continue
        h_full, w_full = frame.shape[:2]
        kxy = res.keypoints.xy.cpu().numpy()
        kconf = (
            res.keypoints.conf.cpu().numpy()
            if res.keypoints.conf is not None
            else np.ones(kxy.shape[:2])
        )
        for person_xy, person_conf in zip(kxy, kconf):
            face_kp = person_xy[:5]              # nose, l_eye, r_eye, l_ear, r_ear
            face_conf = person_conf[:5]
            valid = face_kp[face_conf > 0.3]
            if len(valid) < 2:
                continue
            x1, y1 = valid.min(axis=0)
            x2, y2 = valid.max(axis=0)
            bw, bh = (x2 - x1), (y2 - y1)
            side = max(bw, bh, 1.0)
            cx_p, cy_p = (x1 + x2) / 2, (y1 + y2) / 2
            half = side * (1.0 + pad) / 2
            fx1 = max(0.0, cx_p - half) / w_full
            fy1 = max(0.0, cy_p - half) / h_full
            fx2 = min(float(w_full), cx_p + half) / w_full
            fy2 = min(float(h_full), cy_p + half) / h_full
            detections.append((float(fx1), float(fy1), float(fx2 - fx1), float(fy2 - fy1)))
    return detections


def _cluster_webcam(detections, n_sampled, stability_threshold: float = 0.4):
    """DBSCAN trên center của bbox; chọn cluster ổn định nhất + gần edge."""
    from sklearn.cluster import DBSCAN

    if len(detections) < 5:
        return None
    centers = np.array([(x + w / 2, y + h / 2) for x, y, w, h in detections])
    labels = DBSCAN(eps=0.05, min_samples=5).fit_predict(centers)

    best_cluster, best_score = None, -1.0
    for label in set(labels) - {-1}:
        mask = labels == label
        cluster_size = int(mask.sum())
        stability = cluster_size / n_sampled
        if stability < stability_threshold:
            continue
        cluster_centers = centers[mask]
        cx, cy = cluster_centers.mean(axis=0)
        edge_dist = min(cx, cy, 1.0 - cx, 1.0 - cy)
        score = stability - edge_dist   # ưu tiên cụm gần rìa (webcam thường ở góc)
        if score > best_score:
            best_score = score
            best_cluster = label

    if best_cluster is None:
        return None

    mask = labels == best_cluster
    cluster = [detections[i] for i, m in enumerate(mask) if m]
    xmin = float(np.mean([d[0] for d in cluster]))
    ymin = float(np.mean([d[1] for d in cluster]))
    width = float(np.mean([d[2] for d in cluster]))
    height = float(np.mean([d[3] for d in cluster]))
    cx, cy = xmin + width / 2, ymin + height / 2
    return _BBox(
        xmin=xmin, ymin=ymin, width=width, height=height,
        stability_score=float(mask.sum()) / n_sampled,
        edge_distance=float(min(cx, cy, 1.0 - cx, 1.0 - cy)),
    )


# --- Setup ---
if not _ensure_ultralytics():
    raise RuntimeError("Không cài được ultralytics. Bật Internet trong Settings của Kaggle Notebook.")

from ultralytics import YOLO

# yolov8n-pose.pt sẽ tự download từ Ultralytics CDN nếu chưa có
print("Loading yolov8n-pose.pt (auto-download nếu cần)...")
yolo = YOLO("yolov8n-pose.pt")
print(f"Device: {yolo.device}")

# --- Detection loop ---
webcam_bboxes = {}
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    try:
        frames = _sample_frames(Path(vp), n_frames=30)
        if not frames:
            webcam_bboxes[clip_id] = None
            print(f"  {clip_id}: không đọc được frame")
            continue
        dets = _pose_detect_faces(yolo, frames, conf=0.3)
        bbox = _cluster_webcam(dets, n_sampled=len(frames), stability_threshold=0.4)
        if bbox:
            webcam_bboxes[clip_id] = asdict(bbox)
            print(
                f"  {clip_id}: x={bbox.xmin:.2f} y={bbox.ymin:.2f} "
                f"w={bbox.width:.2f} h={bbox.height:.2f} "
                f"(stab={bbox.stability_score:.2f})"
            )
        else:
            webcam_bboxes[clip_id] = None
            print(f"  {clip_id}: không phát hiện webcam ổn định ({len(dets)} faces)")
    except Exception as e:
        webcam_bboxes[clip_id] = None
        print(f"  {clip_id}: lỗi {e}")

# --- Save ---
bbox_path = os.path.join(PROC_DIR, "webcam_bboxes.json")
with open(bbox_path, "w", encoding="utf-8") as f:
    json.dump(webcam_bboxes, f, indent=2)

n_detected = sum(1 for v in webcam_bboxes.values() if v)
print(f"Webcam bboxes -> {bbox_path}")
print(f"  Phát hiện được: {n_detected}/{len(webcam_bboxes)} clip")


Loading yolov8n-pose.pt (auto-download nếu cần)...
Device: cpu
  raw_001_seg_001: x=0.39 y=0.17 w=0.32 h=0.56 (stab=1.00)
  raw_001_seg_002: x=0.40 y=0.17 w=0.31 h=0.55 (stab=1.00)
  raw_001_seg_003: x=0.41 y=0.16 w=0.31 h=0.55 (stab=1.00)
  raw_001_seg_004: x=0.42 y=0.14 w=0.30 h=0.53 (stab=1.00)
  raw_001_seg_005: x=0.43 y=0.15 w=0.30 h=0.53 (stab=1.00)
  raw_001_seg_006: x=0.42 y=0.16 w=0.31 h=0.55 (stab=1.00)
  raw_001_seg_007: x=0.41 y=0.16 w=0.30 h=0.54 (stab=1.00)
  raw_001_seg_008: x=0.41 y=0.18 w=0.32 h=0.57 (stab=1.00)
  raw_001_seg_009: x=0.41 y=0.19 w=0.32 h=0.57 (stab=1.00)
  raw_001_seg_010: x=0.41 y=0.19 w=0.33 h=0.59 (stab=1.00)
  raw_001_seg_011: x=0.42 y=0.18 w=0.32 h=0.57 (stab=1.00)
  raw_001_seg_012: x=0.42 y=0.20 w=0.32 h=0.56 (stab=1.00)
  raw_001_seg_013: x=0.42 y=0.19 w=0.32 h=0.57 (stab=1.00)
  raw_001_seg_014: x=0.41 y=0.18 w=0.32 h=0.56 (stab=1.00)
  raw_001_seg_015: x=0.45 y=0.12 w=0.28 h=0.50 (stab=1.00)
  raw_001_seg_016: x=0.46 y=0.13 w=0.28 h=0.50 (stab

## Bước 3 — Annotation Pipeline

Mỗi agent chạy tuần tự: **tải → xử lý toàn bộ → unload** để tiết kiệm VRAM (T4 16GB).

| Agent | Model | VRAM | Thời gian |  
|-------|-------|------|----------|
| Whisper ASR | whisper-large-v3 | ~3 GB | ~30s/clip |
| OpenFace AUs | binary | CPU | ~10s/clip |
| Qwen-VL | VL-7B 4bit | ~7 GB | ~20s/clip |
| Qwen-Audio | Audio-7B 4bit | ~5 GB | ~15s/clip |
| Consolidator | 7B 4bit | ~5 GB | ~30s/clip |

In [18]:
# ============================================================
# CELL 10 -- Phase 1: ASR Transcription (v2 bilingual)
# ============================================================
# Backend duoc chon tu CELL 2: ASR_BACKEND (whisper | phowhisper)
# v2: bilingual routing (vi/en), fastText LID cross-check
from types import SimpleNamespace
from vie_gameemo.data.annotator.whisper_asr import build_asr, transcribe_clip

# v2: per-language config, metadata-first routing
asr_cfg = SimpleNamespace(
    backend=ASR_BACKEND,
    language_routing='metadata',        # metadata | auto | force
    detect_for_validation=True,         # always run fastText LID for audit
    lang_prob_threshold=0.6,
    text_lid=SimpleNamespace(backend='fasttext', model='lid.176.ftz'),
    whisper=SimpleNamespace(
        model_name=WHISPER_MODEL,
        compute_type='int8_float16',
        vad_filter=True,
        no_speech_threshold=0.45,
        beam_size=5,
        condition_on_previous_text=False,
        vi=SimpleNamespace(language='vi', initial_prompt=None, post_process='bartpho'),
        en=SimpleNamespace(language='en', initial_prompt=None, post_process='none'),
    ),
    phowhisper=SimpleNamespace(
        model_name=PHOWHISPER_MODEL,
        compute_type='float16',
        chunk_length_s=30,
        batch_size=8,
    ),
    bartpho=SimpleNamespace(
        enabled=USE_BARTPHO,
        model_name=BARTPHO_MODEL,
        max_length=256,
        num_beams=4,
        prefix='Sua loi chinh ta va hoan thien cau: ',
    ),
)

print(f"Loading ASR ({ASR_BACKEND})...")
asr_inst, bartpho_inst = build_asr(asr_cfg)
asr_inst.load()
if bartpho_inst is not None:
    bartpho_inst.load()
    print(f"BARTpho loaded: {BARTPHO_MODEL}")

transcripts = {}
asr_metadata = {}  # v2: store language detection metadata
empty_count = 0
audio_dir = os.path.join(PROC_DIR, 'audios')
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_path = os.path.join(audio_dir, f'{clip_id}.wav')
    # v2: source_language from CLIP_LABELS or default vi
    entry = CLIP_LABELS.get(clip_id)
    source_lang = entry.get('source_language', 'vi') if isinstance(entry, dict) else 'vi'
    if os.path.exists(audio_path):
        result = transcribe_clip(asr_inst, bartpho_inst, Path(audio_path),
                                 source_language=source_lang, asr_cfg=asr_cfg)
        transcripts[clip_id] = result.text
        asr_metadata[clip_id] = {
            'asr_detected_language': result.asr_detected_language,
            'text_detected_language': result.text_detected_language,
            'language_detect_confidence': result.language_detect_confidence,
            'language_mismatch': result.language_mismatch,
        }
        if not result.text:
            empty_count += 1
            print(f'  {clip_id}: [im lang]')
        elif result.language_mismatch:
            print(f'  {clip_id}: LID mismatch! source={source_lang}, detected={result.text_detected_language}')
    else:
        print(f'  {clip_id}: audio not found, skipping')

asr_inst.unload()
if bartpho_inst is not None:
    bartpho_inst.unload()

print(f"
✅ Transcribed {len(transcripts)} clips ({empty_count} empty/silent)")
if asr_metadata:
    mismatch_count = sum(1 for m in asr_metadata.values() if m.get('language_mismatch'))
    print(f"   Language mismatches: {mismatch_count}/{len(asr_metadata)}")


In [19]:
%%writefile /kaggle/working/vie-gameemo-skeleton/src/vie_gameemo/data/annotator/openface_au.py
"""OpenFace Action Unit extraction.

Wraps the OpenFace 2.x binary (FeatureExtraction) to extract per-frame
Action Unit (AU) intensities, which are used by:
- Peak frame detector (find frame with max emotional expression)
- Final annotation (Cved — visual expression description)

Requires OpenFace 2.x installed; see https://github.com/TadasBaltrusaitis/OpenFace
Configure binary path in config.annotation.openface.binary_path.
"""

import csv
import logging
import subprocess
import tempfile
from pathlib import Path

logger = logging.getLogger(__name__)

_DEFAULT_TARGET_AUS = [1, 2, 4, 5, 6, 7, 12, 15, 17, 20, 23, 25, 26, 45]


def extract_aus(
    video_path: Path,
    openface_binary: Path,
    output_dir: Path,
    target_aus: list[int] | None = None,
) -> dict[int, list[float]]:
    """Extract Action Unit intensities per frame using OpenFace.

    Args:
        video_path: Input video file.
        openface_binary: Path to OpenFace FeatureExtraction executable.
        output_dir: Where to save OpenFace CSV output.
        target_aus: AU codes to keep (e.g., [1, 2, 4, 6, 12]). None = default set.

    Returns:
        Dict mapping AU code → list of per-frame intensities.

    Raises:
        FileNotFoundError: If video or binary missing.
        RuntimeError: If OpenFace fails.
    """
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    if not openface_binary.exists():
        raise FileNotFoundError(
            f"OpenFace binary not found: {openface_binary}. "
            "See https://github.com/TadasBaltrusaitis/OpenFace for installation."
        )

    target_aus = target_aus or _DEFAULT_TARGET_AUS
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"{video_path.stem}.csv"
    if csv_path.exists():
        logger.debug("AU CSV already exists (skip): %s", csv_path)
        return _parse_au_csv(csv_path, target_aus)

    cmd = [
        str(openface_binary),
        "-f", str(video_path),
        "-out_dir", str(output_dir),
        "-aus",
    ]
    # OpenFace resolves its model directory ("model/") relative to the
    # binary's location (i.e. build/bin/model after a standard CMake build).
    # Run from that directory so model lookup is unambiguous.
    run_cwd = openface_binary.parent
    model_dir = run_cwd / "model"
    if not model_dir.exists():
        raise RuntimeError(
            f"OpenFace model directory missing: {model_dir}. "
            "Did download_models.sh complete? "
            f"Listing of {run_cwd}: {sorted(p.name for p in run_cwd.iterdir()) if run_cwd.exists() else '<missing>'}"
        )

    logger.info("Running OpenFace on %s (cwd=%s)", video_path.name, run_cwd)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(run_cwd))

    # OpenFace sometimes exits non-zero but still writes a valid CSV; only
    # treat as failure if the CSV is also missing. Surface stdout+stderr
    # because OpenFace writes most diagnostics to stdout.
    if not csv_path.exists():
        diag = ((result.stderr or "") + "\n" + (result.stdout or "")).strip() or "(no output)"
        raise RuntimeError(
            f"OpenFace failed on {video_path} (rc={result.returncode}, cwd={run_cwd}): "
            f"{diag[:800]}"
        )
    if result.returncode != 0:
        logger.warning(
            "OpenFace returned rc=%d on %s but CSV exists — continuing. stderr=%r",
            result.returncode, video_path.name, (result.stderr or "")[:200],
        )

    return _parse_au_csv(csv_path, target_aus)


def aggregate_au_intensity(au_intensities: dict[int, list[float]]) -> list[float]:
    """Sum AU intensities per frame across all target AUs.

    Args:
        au_intensities: AU code → per-frame intensity list.

    Returns:
        List of per-frame aggregated scores (higher = more expression).
    """
    if not au_intensities:
        return []

    n_frames = max(len(v) for v in au_intensities.values())
    aggregated = [0.0] * n_frames
    for intensities in au_intensities.values():
        for i, val in enumerate(intensities):
            if i < n_frames:
                aggregated[i] += val
    return aggregated


def _parse_au_csv(
    csv_path: Path,
    target_aus: list[int],
) -> dict[int, list[float]]:
    """Parse OpenFace AU intensity CSV output.

    Args:
        csv_path: Path to OpenFace output CSV.
        target_aus: AU codes to extract.

    Returns:
        Dict of AU code → list of intensities.
    """
    result: dict[int, list[float]] = {au: [] for au in target_aus}

    with open(csv_path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            for au in target_aus:
                col_key = f" AU{au:02d}_r"
                alt_key = f"AU{au:02d}_r"
                val_str = row.get(col_key, row.get(alt_key, "0")).strip()
                try:
                    result[au].append(float(val_str))
                except ValueError:
                    result[au].append(0.0)

    return result



Overwriting /kaggle/working/vie-gameemo-skeleton/src/vie_gameemo/data/annotator/openface_au.py


In [30]:
# ============================================================
# CELL FIX — Tải và copy model OpenFace bị thiếu
# ============================================================
import os

print("🚀 Đang tải lại và sửa lỗi thiếu model OpenFace...")

OPENFACE_DIR = '/kaggle/working/OpenFace'
BIN_DIR = f'{OPENFACE_DIR}/build/bin'

# 1. Chạy lại script tải model (sẽ mất khoảng 1-2 phút)
!cd {OPENFACE_DIR} && bash download_models.sh

# 2. Copy folder model vào thư mục bin để tool đọc được
!cp -r {OPENFACE_DIR}/lib/local/LandmarkDetector/model {BIN_DIR}/

# 3. Copy folder classifiers (chứa file HAAR) vào thư mục bin
!cp -r {OPENFACE_DIR}/lib/3rdParty/OpenCV/classifiers {BIN_DIR}/

🚀 Đang tải lại và sửa lỗi thiếu model OpenFace...
--2026-05-23 03:14:30--  https://www.dropbox.com/s/7na5qsjzz8yfoer/cen_patches_0.25_of.dat
Resolving www.dropbox.com (www.dropbox.com)... 162.125.9.18, 2620:100:601f:18::a27d:912
Connecting to www.dropbox.com (www.dropbox.com)|162.125.9.18|:443... 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/nbdylr41jbtpykft3m56s/cen_patches_0.25_of.dat?rlkey=r9juzsg39pn40wvv0seyfddz4 [following]
--2026-05-23 03:14:30--  https://www.dropbox.com/scl/fi/nbdylr41jbtpykft3m56s/cen_patches_0.25_of.dat?rlkey=r9juzsg39pn40wvv0seyfddz4
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucd09787cca15e15fa2a41779b5f.dl.dropboxusercontent.com/cd/0/inline/DA8N1QPapozXA5vRw2rQhIAXH8JS_kxV52zks-ursLk78YAIfJQqgl0qEW19mAwIhTadpHvQeJxs26xe-uvKtsY_mF0FDmvIrZp-MZba3ImsR6vnVtlljebCNFa-ZpQj0yo/file# [following]
--2026-05-23 03:14:30--  https://ucd09787cca15e15fa2a41779b5f.dl.dropboxusercontent.com/cd/0/inline/DA8N1QPapozXA5vRw2rQhIAXH8JS_kxV52zks-ursLk78YAIfJQqgl0qEW19mAwIhTadpHvQeJxs26xe-uvKtsY_mF0FDmvIrZp-MZba3ImsR6vnVtlljebCNFa-ZpQj0yo/file
Resolving ucd09787cca15e15fa2a41779b5f.dl.dropboxusercontent.com (ucd09787cca15e15fa2a41779b5f.dl.dr

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



✅ Đã fix xong lỗi thiếu file! Bạn hãy chạy lại Cell 11 nhé.


In [31]:
# ============================================================
# CELL 11 — Phase 2: AU Extraction (OpenFace hoặc MediaPipe fallback)
# ============================================================
# OpenFace model files (CEN patch experts / HAAR) missing on Kaggle build
# -> force MediaPipe fallback. Remove this line once download_models.sh succeeds.
USE_OPENFACE = True

from pathlib import Path

au_results = {}

if USE_OPENFACE:
    import importlib, vie_gameemo.data.annotator.openface_au as _of
    importlib.reload(_of)
    au_tmp = Path(WORKING) / 'openface_tmp'
    au_tmp.mkdir(parents=True, exist_ok=True)
    openface_bin = Path(OPENFACE_BINARY)
    for vp in video_paths:
        vp_path = Path(vp)
        clip_id = vp_path.stem
        try:
            aus = _of.extract_aus(vp_path, openface_bin, au_tmp)
            intensity = _of.aggregate_au_intensity(aus)
            # Tóm tắt thành string cho LLM
            top_aus = sorted(aus.items(), key=lambda x: -max(x[1]) if x[1] else 0)[:5]
            au_str = ', '.join(f'AU{k}={max(v):.1f}' for k, v in top_aus if v)
            au_results[clip_id] = au_str or 'N/A'
            print(f'  {clip_id}: {au_str}')
        except Exception as e:
            au_results[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
else:
    # Fallback: phát hiện mặt bằng OpenCV Haar cascade (mediapipe 0.10+ không còn mp.solutions)
    print('Dùng OpenCV Haar cascade fallback cho face detection...')
    import cv2

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    for vp in video_paths:
        clip_id = os.path.splitext(os.path.basename(vp))[0]
        frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
        frame_files = sorted(f for f in os.listdir(frames_dir) if f.endswith('.jpg'))[:8]

        au_desc = 'N/A'
        if frame_files:
            mid_frame = cv2.imread(os.path.join(frames_dir, frame_files[len(frame_files)//2]))
            gray = cv2.cvtColor(mid_frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
            if len(faces) > 0:
                au_desc = f'Face detected ({len(faces)} via Haar, no AU values — OpenFace needed for exact AUs)'
            else:
                au_desc = 'No face detected'
        au_results[clip_id] = au_desc
        print(f'  {clip_id}: {au_desc}')

print('✅ AU extraction xong')


  raw_001_seg_001: AU26=3.1, AU7=2.3, AU25=1.9, AU17=1.7, AU45=1.6
  raw_001_seg_002: AU15=4.9, AU25=3.2, AU7=3.0, AU17=2.8, AU12=2.1
  raw_001_seg_003: AU25=3.0, AU7=2.8, AU26=2.5, AU45=2.2, AU17=2.1
  raw_001_seg_004: AU25=2.5, AU7=2.3, AU4=2.1, AU15=2.0, AU26=1.8
  raw_001_seg_005: AU4=2.2, AU7=2.0, AU20=1.7, AU45=1.6, AU26=1.5
  raw_001_seg_006: AU23=1.8, AU4=1.7, AU26=1.6, AU45=1.6, AU7=1.6
  raw_001_seg_007: AU7=2.6, AU17=2.1, AU4=2.0, AU26=1.8, AU15=1.4
  raw_001_seg_008: AU7=3.9, AU25=2.0, AU6=1.8, AU12=1.7, AU26=1.7
  raw_001_seg_009: AU7=3.3, AU17=2.3, AU12=2.3, AU6=2.2, AU26=2.2
  raw_001_seg_010: AU7=3.3, AU25=1.6, AU6=1.5, AU12=1.5, AU17=1.3
  raw_001_seg_011: AU7=3.4, AU12=2.6, AU6=2.6, AU25=2.1, AU26=1.3
  raw_001_seg_012: AU7=3.0, AU12=1.5, AU6=1.1, AU45=1.0, AU26=0.6
  raw_001_seg_013: AU7=3.6, AU12=2.6, AU25=2.1, AU6=1.9, AU1=1.1
  raw_001_seg_014: AU7=3.2, AU26=2.7, AU25=2.3, AU12=2.0, AU17=1.7
  raw_001_seg_015: AU20=2.5, AU7=2.4, AU26=2.0, AU1=1.8, AU4=1.5
  raw_00

In [27]:
# ============================================================
# CELL 12 — Phase 3: Qwen-VL Visual Descriptions
# ============================================================
import os
import gc
import torch
from pathlib import Path
from vie_gameemo.data.annotator.qwen_vl_agent import QwenVLAgent

print(f'Loading Qwen-VL: {QWEN_VL_MODEL}...')

# Thêm prompt phù hợp với mục tiêu dự án của bạn
VL_PROMPT = "Describe this video game frame in detail. Focus on the environment, the main action taking place, the lighting, and the overall atmosphere or mood (e.g., tense, scary, peaceful, chaotic)."

vl_agent = QwenVLAgent(
    model_name=QWEN_VL_MODEL,
    quantization=QUANTIZATION,
    prompt=VL_PROMPT  # <--- Truyền prompt vào đây
)
vl_agent.load()

visual_descriptions = {}
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
    frame_files = sorted(
        [Path(frames_dir) / f for f in os.listdir(frames_dir) if f.endswith('.jpg')]
    )[:4]   # Dùng 4 frames để mô tả

    if frame_files:
        try:
            descs = vl_agent.batch_describe(frame_files)
            visual_descriptions[clip_id] = ' | '.join(descs)
            print(f'  {clip_id}: "{descs[0][:100]}..."')
        except Exception as e:
            visual_descriptions[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
    else:
        visual_descriptions[clip_id] = 'N/A'

# Unload
vl_agent.unload()
del vl_agent
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Qwen-VL unloaded')

Loading Qwen-VL: Qwen/Qwen2.5-VL-7B-Instruct...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

  raw_001_seg_001: "This image appears to be a screenshot from a live stream or a recorded video of someone participatin..."
  raw_001_seg_002: "This image appears to be a screenshot from a live stream or a recorded video session of a streamer e..."
  raw_001_seg_003: "This image appears to be a screenshot from a live stream or a recorded video of someone playing a vi..."
  raw_001_seg_004: "This image appears to be a screenshot from a live-streamed gaming session or a similar interactive o..."
  raw_001_seg_005: "This image appears to be a screenshot from a live stream or a recorded video session, likely involvi..."
  raw_001_seg_006: "This image appears to be a still from a live-streamed gaming session or a similar interactive online..."
  raw_001_seg_007: "This image appears to be a screenshot from a live stream or a recorded gaming session. Here's a deta..."
  raw_001_seg_008: "This image appears to be a screenshot from a live stream or a recorded video of a gaming session. He..."


In [34]:
# ============================================================
# CELL 13 — Phase 4: Qwen-Audio Descriptions
# ============================================================
import os
import gc
import torch
from pathlib import Path
from vie_gameemo.data.annotator.qwen_audio_agent import QwenAudioAgent

print(f'Loading Qwen-Audio: {QWEN_AUDIO_MODEL}...')

# Khai báo prompt để hướng dẫn Qwen-Audio tập trung vào cảm xúc và tiếng động môi trường
AUDIO_PROMPT = "Based on the provided audio input, describe the speaker's tone, background sound effects, and overall auditory atmosphere."
audio_agent = QwenAudioAgent(
    model_name=QWEN_AUDIO_MODEL,
    quantization=QUANTIZATION,
    prompt=AUDIO_PROMPT  # <--- Đã bổ sung tham số bị thiếu ở đây
)
audio_agent.load()

audio_descriptions = {}
audio_dir = os.path.join(PROC_DIR, 'audios')
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_path = Path(os.path.join(audio_dir, f'{clip_id}.wav'))
    if audio_path.exists():
        try:
            desc = audio_agent.batch_describe([audio_path])[0]
            audio_descriptions[clip_id] = desc
            print(f'  {clip_id}: "{desc[:100]}..."')
        except Exception as e:
            audio_descriptions[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
    else:
        audio_descriptions[clip_id] = 'N/A'

# Unload
audio_agent.unload()
del audio_agent
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Qwen-Audio unloaded')

Loading Qwen-Audio: Qwen/Qwen2-Audio-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

  raw_001_seg_001: "The speaker's tone is calm and soothing, characterized by a medium pitch and a slow tempo of speech...."
  raw_001_seg_002: "The speaker's tone is calm and soothing, characterized by a low pitch that creates a serene atmosphe..."
  raw_001_seg_003: "The speaker's tone is calm and composed, characterized by a medium pitch and a steady pace of speech..."
  raw_001_seg_004: "The speaker's tone is calm and soothing, characterized by a medium pitch and a slow tempo of speech...."
  raw_001_seg_005: "The speaker's tone is calm and composed, characterized by a medium pitch and a steady pace of speech..."
  raw_001_seg_006: "The speaker's tone is calm and soothing, characterized by a medium pitch and a slow tempo of speakin..."
  raw_001_seg_007: "The speaker's tone is calm and composed, characterized by a medium pitch and a steady pace of speech..."
  raw_001_seg_008: "The speaker's tone is calm and composed, characterized by a medium pitch and a steady pace of speech..."


In [47]:
# ============================================================
# CELL 14 — Phase 5: Consolidator (reasoning + nhãn dự đoán + confidence)
# ============================================================
# Notebook xây prompt riêng (yêu cầu thêm <answer>, <confidence>, <distribution>)
# và dùng Consolidator._generate() để inference.
import json
import re
from vie_gameemo.data.annotator.consolidator import Consolidator

ENABLE_THINKING = 'Qwen3' in ANNOTATION_MODEL   # Qwen3 hỗ trợ /think token
print(f'Loading Consolidator: {ANNOTATION_MODEL} (thinking={ENABLE_THINKING})...')

consolidator = Consolidator(
    model_name=ANNOTATION_MODEL,
    quantization=QUANTIZATION,
    max_new_tokens=500,
)
consolidator.load()

CONSOLIDATE_PROMPT_TEMPLATE = """\
Bạn là chuyên gia phân tích cảm xúc đa phương thức cho streamer Việt Nam chơi game.
Dựa trên bằng chứng đa modal sau đây, hãy:
1) Viết reasoning ngắn 3-5 câu bằng tiếng Việt.
2) Chọn 1 nhãn cảm xúc duy nhất từ tập: {valid_labels}.
3) Đưa ra confidence (0.0-1.0) — mức độ chắc chắn của bạn với nhãn đó.
4) Đưa ra phân phối xác suất TOP-3 nhãn khả dĩ (dạng JSON {{"label": prob}}).
   Tổng xác suất TOP-3 phải = 1.0.

Nhãn do người dùng cung cấp (tham khảo, có thể đúng/sai): {emotion_label}
Confidence của người gán nhãn: {manual_confidence:.2f}

Bằng chứng đa phương thức:
- Khuôn mặt (Action Units): {face_aus}
- Cảnh và bối cảnh: {visual_objective}
- Đặc điểm giọng nói: {audio_tone}
- Lời nói (transcript): "{transcript}"

Hướng dẫn:
- Liên kết các bằng chứng từ nhiều modality
- Nếu có slang gaming tiếng Anh, hãy giải thích nghĩa
- KHÔNG bịa thông tin không có trong bằng chứng
- Nếu bằng chứng yếu/mâu thuẫn, để confidence thấp (vd 0.3-0.5)
- Ngôn ngữ: Tiếng Việt

Output theo format CHÍNH XÁC (không thêm chữ nào ngoài 4 thẻ này):
<think>
[Reasoning của bạn ở đây — 3-5 câu]
</think>
<answer>nhãn_dự_đoán</answer>
<confidence>0.XX</confidence>
<distribution>{{"label_a": 0.XX, "label_b": 0.XX, "label_c": 0.XX}}</distribution>
"""

_THINK_RE      = re.compile(r'<think>(.*?)</think>', re.DOTALL)
_ANSWER_RE     = re.compile(r'<answer>\s*(\w+)\s*</answer>', re.IGNORECASE)
_CONF_RE       = re.compile(r'<confidence>\s*([0-9.]+)\s*</confidence>', re.IGNORECASE)
_DIST_RE       = re.compile(r'<distribution>\s*(\{.*?\})\s*</distribution>', re.DOTALL | re.IGNORECASE)


def _parse_consolidator_output(text: str, manual_label: str) -> dict:
    """Parse output thành dict {reasoning, label, confidence, distribution}."""
    reasoning = ''
    m = _THINK_RE.search(text)
    if m:
        reasoning = m.group(1).strip()

    label = manual_label
    m = _ANSWER_RE.search(text)
    if m and m.group(1).lower() in VALID_LABELS:
        label = m.group(1).lower()

    confidence = 0.5
    m = _CONF_RE.search(text)
    if m:
        try:
            confidence = max(0.0, min(1.0, float(m.group(1))))
        except ValueError:
            pass

    distribution: dict[str, float] = {}
    m = _DIST_RE.search(text)
    if m:
        try:
            raw = json.loads(m.group(1))
            distribution = {k: float(v) for k, v in raw.items() if k in VALID_LABELS}
            s = sum(distribution.values())
            if s > 0:
                distribution = {k: v / s for k, v in distribution.items()}
        except (json.JSONDecodeError, TypeError, ValueError):
            distribution = {}

    return {
        'reasoning': reasoning,
        'label': label,
        'confidence': confidence,
        'distribution': distribution,
        'raw_output': text,
    }


reasoning_results = {}

for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    manual = manual_label_meta.get(clip_id, {'label': 'neutral', 'confidence': 0.0})

    prompt = CONSOLIDATE_PROMPT_TEMPLATE.format(
        valid_labels=', '.join(VALID_LABELS),
        emotion_label=manual['label'],
        manual_confidence=manual['confidence'],
        face_aus=au_results.get(clip_id, 'N/A'),
        visual_objective=visual_descriptions.get(clip_id, 'N/A'),
        audio_tone=audio_descriptions.get(clip_id, 'N/A'),
        transcript=transcripts.get(clip_id, ''),
    )

    try:
        raw_output = consolidator._generate(prompt)
        parsed = _parse_consolidator_output(raw_output, manual['label'])
    except Exception as e:
        parsed = {
            'reasoning': '', 'label': manual['label'], 'confidence': 0.0,
            'distribution': {}, 'raw_output': f'ERROR: {e}',
        }
        print(f'  {clip_id}: ⚠️  {e}')

    reasoning_results[clip_id] = parsed
    print(f'  {clip_id} [manual={manual["label"]}/{manual["confidence"]:.2f}] '
          f'→ pred={parsed["label"]}/{parsed["confidence"]:.2f}: '
          f'{parsed["reasoning"][:60]}...')

consolidator.unload()
del consolidator
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Consolidation xong (mỗi clip có reasoning + label + confidence + distribution)')

Loading Consolidator: Qwen/Qwen2.5-7B-Instruct (thinking=False)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  raw_001_seg_001 [manual=neutral/0.50] → pred=neutral/0.85: The individual in the image appears to be in a dedicated gam...
  raw_001_seg_002 [manual=neutral/0.50] → pred=amused/0.85: The image shows a streamer in a home studio setting, likely ...
  raw_001_seg_003 [manual=neutral/0.50] → pred=neutral/0.80: Từ các bằng chứng đa phương thức, chúng ta có thể thấy người...
  raw_001_seg_004 [manual=neutral/0.50] → pred=focus/0.85: Ảnh chụp màn hình này xuất hiện từ một cuộc phát sóng trực t...
  raw_001_seg_005 [manual=neutral/0.50] → pred=focus/0.85: The individual in the image is seated in a gaming chair, ind...
  raw_001_seg_006 [manual=neutral/0.50] → pred=focus/0.80: The image shows a person in a dedicated gaming setup, indica...
  raw_001_seg_007 [manual=neutral/0.50] → pred=focus/0.85: The individual in the screenshot is seated in a gaming chair...
  raw_001_seg_008 [manual=neutral/0.50] → pred=focus/0.85: The streamer is seated in a gaming chair with a setup typica...
  raw_001_s

## Bước 4 — Lưu Annotations

In [48]:
# ============================================================
# CELL 15 — Tạo Annotation JSON cho từng clip
# Bao gồm: nhãn thủ công + confidence thủ công + nhãn dự đoán + confidence dự đoán
# + phân phối xác suất TOP-3 từ Consolidator + thông tin segment 3–7s.
# ============================================================
import json
from datetime import datetime

saved = []
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    manual = manual_label_meta.get(clip_id, {
        'label': 'neutral', 'confidence': 0.0, 'alternatives': {}, 'is_placeholder': True,
    })
    pred = reasoning_results.get(clip_id, {
        'reasoning': '', 'label': manual['label'], 'confidence': 0.0, 'distribution': {},
    })
    seg_info = segment_meta.get(clip_id, {})

    # Nhãn cuối cùng = LUÔN dùng output của Consolidator (CELL 14).
    # (Trước đây ưu tiên manual; đổi thành nghe theo model 100% theo yêu cầu.)
    final_label = pred['label']
    final_confidence = pred['confidence']
    label_source = 'model'

    annotation = {
        'clip_id': clip_id,
        'video_path': vp,
        # --- thông tin segment (3–7s) ---
        'segment': {
            'raw_video': seg_info.get('raw_video'),
            'start_sec': seg_info.get('start_sec'),
            'duration_sec': seg_info.get('duration_sec'),
        },
        # --- nhãn cuối cùng (dùng cho training) ---
        'emotion_label': final_label,
        'confidence': round(final_confidence, 4),
        'label_source': label_source,
        # --- chi tiết nhãn thủ công ---
        'manual_label': {
            'label': manual['label'],
            'confidence': round(manual['confidence'], 4),
            'alternatives': {k: round(v, 4) for k, v in manual['alternatives'].items()},
            'is_placeholder': manual['is_placeholder'],
        },
        # --- chi tiết nhãn do Consolidator dự đoán ---
        'predicted_label': {
            'label': pred['label'],
            'confidence': round(pred['confidence'], 4),
            'distribution': {k: round(v, 4) for k, v in pred['distribution'].items()},
        },
        # --- bằng chứng đa modal ---
        'transcript': transcripts.get(clip_id, ''),
        'face_aus': au_results.get(clip_id, 'N/A'),
        'visual_objective': visual_descriptions.get(clip_id, 'N/A'),
        'audio_tone': audio_descriptions.get(clip_id, 'N/A'),
        'reasoning': pred['reasoning'],
        # --- metadata ---
        'annotation_model': ANNOTATION_MODEL,
        'created_at': datetime.now().isoformat(),
        'split': 'train',   # Sẽ được phân chia lại trong training notebook
    }

    out_path = os.path.join(ANNOT_DIR, f'{clip_id}.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(annotation, f, ensure_ascii=False, indent=2)
    saved.append(out_path)

print(f'✅ Đã lưu {len(saved)} annotation files → {ANNOT_DIR}')
for p in saved[:3]:
    print(f'  {p}')

✅ Đã lưu 20 annotation files → /kaggle/working/data/annotations
  /kaggle/working/data/annotations/raw_001_seg_001.json
  /kaggle/working/data/annotations/raw_001_seg_002.json
  /kaggle/working/data/annotations/raw_001_seg_003.json


In [49]:
# ============================================================
# CELL 16 — Thống kê: phân phối nhãn + confidence + agreement manual vs model
# ============================================================
from collections import Counter

# Target % per gaming_8 label (config.yaml: labeling.class_distribution)
TARGET_DISTRIBUTION = {
    'neutral':   0.20,
    'hype':      0.13,
    'amused':    0.11,
    'tilted':    0.11,
    'sad':       0.10,
    'shocked':   0.11,
    'fear':      0.09,
    'disgusted': 0.08,
}

annotations = []
for f in os.listdir(ANNOT_DIR):
    if f.endswith('.json'):
        with open(os.path.join(ANNOT_DIR, f), encoding='utf-8') as fp:
            annotations.append(json.load(fp))

total = len(annotations)
print(f'Tổng annotations: {total}')

# --- Độ dài clip ---
durations = [
    a.get('segment', {}).get('duration_sec') or 0.0
    for a in annotations
]
durations = [d for d in durations if d > 0]
if durations:
    print(f'\nĐộ dài clip: min={min(durations):.2f}s, max={max(durations):.2f}s, '
          f'avg={sum(durations)/len(durations):.2f}s '
          f'(yêu cầu {CLIP_MIN_DURATION}–{CLIP_MAX_DURATION}s)')

# --- Phân phối nhãn ---
label_counts = Counter(a['emotion_label'] for a in annotations)
print('\nPhân phối nhãn  (actual % | target % | gap):')
for label in VALID_LABELS:
    count = label_counts.get(label, 0)
    actual = count / total if total else 0
    target = TARGET_DISTRIBUTION[label]
    gap = actual - target
    bar = '█' * count
    flag = '⚠️ ' if abs(gap) > 0.05 else '   '
    print(f'  {flag}{label:<10}: {bar:<20} {count:>3} '
          f'({actual*100:>4.1f}% | target {target*100:>4.1f}% | {gap*100:+4.1f}%)')

# --- Confidence ---
manual_confs = [a['manual_label']['confidence'] for a in annotations
                if not a['manual_label']['is_placeholder']]
pred_confs   = [a['predicted_label']['confidence'] for a in annotations]
final_confs  = [a['confidence'] for a in annotations]


def _hist(values, bins=(0.0, 0.2, 0.4, 0.6, 0.8, 1.01)):
    edges = list(bins)
    counts = [0] * (len(edges) - 1)
    for v in values:
        for i in range(len(edges) - 1):
            if edges[i] <= v < edges[i + 1]:
                counts[i] += 1
                break
    return counts


def _print_hist(name, values):
    if not values:
        print(f'  {name}: (rỗng)')
        return
    counts = _hist(values)
    edges = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
    avg = sum(values) / len(values)
    print(f'  {name} (n={len(values)}, avg={avg:.3f}):')
    for e, c in zip(edges, counts):
        bar = '█' * c
        print(f'    {e}: {bar:<20} {c}')


print('\nConfidence histogram:')
_print_hist('Manual  ', manual_confs)
_print_hist('Model   ', pred_confs)
_print_hist('Final   ', final_confs)

# --- Agreement giữa nhãn thủ công và nhãn model ---
agree = [a for a in annotations
         if not a['manual_label']['is_placeholder']
         and a['manual_label']['label'] == a['predicted_label']['label']]
n_manual = sum(1 for a in annotations if not a['manual_label']['is_placeholder'])
if n_manual:
    print(f'\nAgreement manual vs model: {len(agree)}/{n_manual} '
          f'({len(agree)/n_manual*100:.1f}%)')

# --- Sanity checks ---
has_reasoning  = sum(1 for a in annotations if a.get('reasoning'))
has_transcript = sum(1 for a in annotations if a.get('transcript'))
has_dist       = sum(1 for a in annotations if a['predicted_label'].get('distribution'))
print(f'\nCó reasoning   : {has_reasoning}/{total}')
print(f'Có transcript  : {has_transcript}/{total}')
print(f'Có distribution: {has_dist}/{total}')

# Cảnh báo clip có confidence thấp — nên review thủ công
LOW_CONF_THRESHOLD = 0.4
low = sorted(
    [(a['clip_id'], a['confidence']) for a in annotations
     if a['confidence'] < LOW_CONF_THRESHOLD],
    key=lambda x: x[1],
)
if low:
    print(f'\n⚠️  {len(low)} clip có final confidence < {LOW_CONF_THRESHOLD} — nên review:')
    for cid, c in low[:10]:
        print(f'    {cid}: {c:.2f}')
    if len(low) > 10:
        print(f'    ... ({len(low) - 10} clip nữa)')

Tổng annotations: 20

Độ dài clip: min=5.00s, max=5.00s, avg=5.00s (yêu cầu 3.0–7.0s)

Phân phối nhãn  (actual % | target % | gap):
     neutral   : ██                     2 (10.0% | target 14.0% | -4.0%)
  ⚠️ focus     : ████████████          12 (60.0% | target 13.0% | +47.0%)
  ⚠️ hype      : ████                   4 (20.0% | target 13.0% | +7.0%)
     amused    : ██                     2 (10.0% | target 11.0% | -1.0%)
  ⚠️ tilted    :                        0 ( 0.0% | target 11.0% | -11.0%)
  ⚠️ sad       :                        0 ( 0.0% | target 10.0% | -10.0%)
  ⚠️ shocked   :                        0 ( 0.0% | target 11.0% | -11.0%)
  ⚠️ fear      :                        0 ( 0.0% | target  9.0% | -9.0%)
  ⚠️ disgusted :                        0 ( 0.0% | target  8.0% | -8.0%)

Confidence histogram:
  Manual   (n=20, avg=0.500):
    0.0-0.2:                      0
    0.2-0.4:                      0
    0.4-0.6: ████████████████████ 20
    0.6-0.8:                      0
    0.8-1

In [50]:
# ============================================================
# CELL 17 — Tạo archive để download (kèm video clips để tham chiếu relative)
# ============================================================
import zipfile
import json as _json

archive_path = os.path.join(WORKING, 'stage0_annotations.zip')

# clip_id -> basename(vp) để rewrite video_path thành relative
clip_basename = {
    os.path.splitext(os.path.basename(vp))[0]: os.path.basename(vp)
    for vp in video_paths
}

n_videos = 0
n_missing = 0
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Annotations — rewrite video_path thành 'clips/<basename>' (relative)
    for f in os.listdir(ANNOT_DIR):
        path = os.path.join(ANNOT_DIR, f)
        if f.endswith('.json'):
            with open(path, encoding='utf-8') as fp:
                ann = _json.load(fp)
            cid = ann.get('clip_id') or f[:-5]
            basename = clip_basename.get(cid) or os.path.basename(ann.get('video_path', ''))
            if basename:
                ann['video_path'] = f'clips/{basename}'
            zf.writestr(f'annotations/{f}', _json.dumps(ann, ensure_ascii=False, indent=2))
        elif f != 'labels.csv':
            zf.write(path, f'annotations/{f}')

    zf.write(LABELS_CSV, 'annotations/labels.csv')

    # Webcam bboxes
    bbox_file = os.path.join(PROC_DIR, 'webcam_bboxes.json')
    if os.path.exists(bbox_file):
        zf.write(bbox_file, 'processed/webcam_bboxes.json')

    # Segment metadata (3–7s)
    seg_file = os.path.join(SEG_DIR, 'segments.json')
    if os.path.exists(seg_file):
        zf.write(seg_file, 'clips/segments.json')

    # Video clips — đóng gói luôn để JSON tham chiếu relative ('clips/...')
    for vp in video_paths:
        if os.path.exists(vp):
            zf.write(vp, f'clips/{os.path.basename(vp)}')
            n_videos += 1
        else:
            n_missing += 1

print(f'✅ Archive: {archive_path}')
print(f'   Size  : {os.path.getsize(archive_path) / 1e6:.1f} MB')
print(f'   Videos: {n_videos} clip ({n_missing} thiếu)')
print()
print('📥 Để download: File browser (bên trái) → tìm stage0_annotations.zip → chuột phải → Download')
print('   Sau khi unzip, video_path trong mỗi JSON trỏ tới clips/<file>.mp4 cùng cấp.')


✅ Archive: /kaggle/working/stage0_annotations.zip
   Size  : 23.6 MB
   Videos: 20 clip (0 thiếu)

📥 Để download: File browser (bên trái) → tìm stage0_annotations.zip → chuột phải → Download
   Sau khi unzip, video_path trong mỗi JSON trỏ tới clips/<file>.mp4 cùng cấp.
